# 05 — Cohort B — cluster, and ship statistics

| | |
|---|---|
| **runs at** | cohort B |
| **reads** | `cohort-B.csv`, `shared_features.csv` |
| **writes** | `cohort-B_edges.csv`, `cohort-B_stats.npz`, `cohort-B_labels.txt`, figures |

Two things happen here, and they answer different questions.

**The standalone answer.** What this cohort finds on its own. This is the
baseline the federated result is compared against in step 10/11 — without it,
there is no way to say federating helped.

**The sufficient statistics.** What leaves for the aggregator. Four matrices,
no rows.

In [ ]:
import os, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Every notebook reads and writes here, so the chain hands work along. In a real
# deployment this directory would be on a different machine for each cohort, and
# only the files named 'what this writes' would ever cross the boundary.
WORK = Path('sle-run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)

def run(cmd):
    """Run one pvclust-py command and echo it, so the notebook shows the command
    line rather than hiding it behind a function."""
    print('$ ' + ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1800:])
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

def show(png):
    """Render a written figure inline. matplotlib only, so this works in CI too."""
    if not Path(png).exists():
        print(f'(missing {png})'); return
    fig, ax = plt.subplots(figsize=(13, 13))
    ax.imshow(plt.imread(png)); ax.axis('off'); ax.set_title(png, fontsize=9)
    plt.show()

In [ ]:
# The SLE data is not in the repository. Download it from Zenodo
# (doi:10.5281/zenodo.20342569), unpack it, and point SLE at the folder.
SLE = Path(os.environ.get('SLE', '../data/SLE_doi.10.5281_zenodo_20342569'))
REAL = (SLE / 'abundance.csv').exists()

if REAL:
    NBOOT, TOPVAR = 1000, 80        # 80 objects keeps p < n in BOTH cohorts
    FEATURE_MAP = str(SLE / 'feature_metadata.txt')
else:
    # A stand-in of the same shape, so every cell runs without the download:
    # two batches of unequal size, a case/control split, and a few proteins
    # measured by more than one reagent.
    NBOOT, TOPVAR = 40, 20
    FEATURE_MAP = 'feature_metadata.txt'
print(f"{'SLE data' if REAL else 'stand-in'}: n_boot={NBOOT}, top_variable={TOPVAR}")

## The standalone clustering

**Both axes are clustered**, always: the proteins across the top and the samples
down the side. They answer different questions and only one of them is on solid
statistical ground.

The bootstrap resamples **rows**. So clustering the proteins asks: drawing
another sample of patients from the same population, would this protein cluster
reappear? That is a well-posed question and the AU *p*-value answers it.

Clustering the samples means resampling proteins, and proteins are co-expressed
rather than independent draws. AU on that axis is **anti-conservative** — a real
number, but an optimistic one. Report it as such.

`minkowski` with `ward.D2` is the pairing carried over from microarray and
RNA-seq work. In pvclust, minkowski means p=2, that is euclidean, because
`dist.pvclust` never forwards `p` to R's `dist()`. Verified against R at maximum
difference 0.0, and it is one of the four distances that pools **exactly** under
federation, which the next step depends on.

In [ ]:
run(['pvclust-py', 'cluster', '--project', 'cohort-B',
     '--matrix', 'cohort-B.csv', '--log2',
     '--metadata', 'cohort-B_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--shared-features', 'shared_features.csv',
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol',
     '--dist', 'minkowski', '--linkage', 'ward.D2',
     '--n-boot', str(NBOOT), '--alpha', '0.95', '--plot'])

In [ ]:
show('cohort-B_dendrogram.png')

## What did it find alone?

**Read `df` before `au`.** `df` is the degrees of freedom of the curve fit: the
number of usable resampling scales minus two. At `df = 0` the fit is degenerate
and the AU beside it means nothing, however confident it looks.

In [ ]:
e = pd.read_csv('cohort-B_edges.csv').sort_values('au', ascending=False)
print(f'{len(e)} clusters evaluated, {(e.au >= 0.95).sum()} at AU >= 0.95')
print()
print(e.head(8)[['n_members', 'au', 'bp', 'se_au', 'df']].to_string(index=False))

## The two-way figure

Both dendrograms, ordered so better-supported clusters come first, with the
demographics tiled down the side. This is also the batch-effect check: if the
samples cluster by plate rather than by biology, the strips show it at a glance.

In [ ]:
run(['pvclust-py', 'heatmap', '--project', 'cohort-B',
     '--matrix', 'cohort-B.csv', '--log2',
     '--metadata', 'cohort-B_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--shared-features', 'shared_features.csv',
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol',
     '--n-boot', str(NBOOT),
     '--annotate', 'Group,Sex,Age_group,Disease_activity,SLEDAI_band',
     '--max-rows', '400', '--max-labels', '160'])

In [ ]:
show('cohort-B_heatmap_pvclust.png')

## What leaves this cohort

Four `p x p` matrices and nothing else:

| | per pair of proteins (i, j) |
|---|---|
| `N` | how many samples have **both** measured |
| `S` | sum of i, over those samples |
| `Q` | sum of i squared, over those samples |
| `G` | sum of i times j, over those samples — the Gram matrix, `XᵀX` |

Those five numbers per pair are exactly what a correlation needs, which is why
they are **sufficient**: the raw rows add nothing further. And every entry is a
sum over rows, which is why they **add** across cohorts.

**The privacy rule is `p < n`.** The rank of `G` is the smaller of n and p, so a
cohort with fewer samples than proteins pins down the exact subspace its rows
occupy, and individuals are hidden only by a rotation inside it. At one sample
`G` has rank one and an eigendecomposition returns that row exactly. A fixed
minimum sample size is the weaker guard; keeping proteins below samples is the
real one. See Homer et al. (2008) for why aggregate summaries are not
automatically safe.

In [ ]:
run(['pvclust-py', 'project-stats', '--project', 'cohort-B',
     '--matrix', 'cohort-B.csv', '--log2',
     '--metadata', 'cohort-B_meta.csv',
     '--adjust', 'combat', '--batch-col', 'Batch', '--protect', 'Group',
     '--shared-features', 'shared_features.csv',
     '--feature-map', FEATURE_MAP, '--feature-label', 'GeneSymbol'])

n = len(pd.read_csv('cohort-B.csv'))
p = len(pd.read_csv('shared_features.csv'))
print(f'\nprivacy check: p={p}, n={n} -> ' + ('OK, p < n' if p < n else '!! p >= n, DO NOT SHIP'))

---
**Next:** `06`. The aggregator sums the statistics from both cohorts.